# Cấu hình B — thêm thời tiết và so sánh với A

**Mục tiêu:** thời tiết cùng giờ có cải thiện dự đoán so với giờ + thứ không?

Theo cấu trúc Iris: chuẩn bị → baseline → mô hình/tuning → validation → lưu kết quả.
Proposal là nguồn yêu cầu. Đây là bước thử nghiệm tiếp nối A tối giản, chưa phải A/B đầy đủ có tháng/ngày lễ.

**A đối chứng:** `hour`, `day_of_week`. **B:** giữ nguyên A, thêm nhiệt độ, mưa, tuyết, mây và nhóm thời tiết.
5 mô hình: Linear Regression, Random Forest, Gradient Boosting, MLP, XGBoost. Ridge không nằm trong lượt này theo lựa chọn của nhóm.
Giả định đã biết thời tiết của giờ dự đoán. Không dùng History, không chấm test.

Bản A cũ ở `experiments/11-11-2026_Huy_ConfigA/`. Vì thiếu dự đoán đầy đủ và provenance của lần cũ, chạy lại A cùng B để đối chứng công bằng; không trộn metrics cũ vào kết quả mới.

## 1. Thiết lập và dữ liệu

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import platform
import subprocess
import sys
import uuid
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

project_dir = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                   if (p / 'src/data.py').is_file())
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
from src.data import DATA_PATH, load_raw, build_hourly_table, split_chronological, fill_missing_weather
from src.features import V1_FEATURES, WEATHER_FEATURES, add_time_features, add_weather_features
from src.models import SEARCH_SPACE, N_RANDOM_TRIALS, RANDOM_STATE, results_table
from src.tuning import describe_folds, run_config
from src.evaluation import regression_metrics, weekday_hour_baseline

MODEL_NAMES = ['Linear Regression', 'Random Forest', 'Gradient Boosting', 'MLP', 'XGBoost']
FEATURE_SETS = {'A_time': list(V1_FEATURES), 'B_time_weather': list(V1_FEATURES) + list(WEATHER_FEATURES)}
run_id = datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
run_dir = project_dir / 'experiments/11-11-2026_Huy_ConfigB' / run_id
print('Lần chạy:', run_id)
display(pd.DataFrame([{'Cấu hình': k, 'Đặc trưng': ', '.join(v)} for k,v in FEATURE_SETS.items()]))

Lần chạy: 20261111_230501_c3b031


,Cấu hình,Đặc trưng
0,A_time,"hour, day_of_week"
1,B_time_weather,"hour, day_of_week, temp_c, rain_1h, snow_1h, c..."


In [2]:
raw = load_raw(DATA_PATH)
hourly = build_hourly_table(raw)
splits = split_chronological(hourly)
assert splits['train'].date_time.max() < splits['validation'].date_time.min()
assert splits['validation'].date_time.max() < splits['test'].date_time.min()
split_summary = pd.DataFrame([
    {'split': name, 'n': len(part), 'start': part.date_time.min(), 'end': part.date_time.max()}
    for name, part in splits.items()
])
display(split_summary)
# Không xử lý hoặc dự đoán test trong thực nghiệm này.
working = {name: splits[name] for name in ('train', 'validation')}
filled = fill_missing_weather(working)
train, validation = [add_weather_features(add_time_features(filled[name]))
                     for name in ('train', 'validation')]
for frame in (train, validation):
    assert frame.date_time.is_unique and frame.date_time.is_monotonic_increasing
    assert np.isfinite(frame[FEATURE_SETS['B_time_weather']].to_numpy(dtype=float)).all()
folds = describe_folds(train)
display(folds)

,split,n,start,end
0,train,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
1,validation,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
2,test,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


,fold,fit_start,fit_end,validation_start,validation_end,n_fit,n_validation
0,1,2012-10-02 09:00:00,2013-08-18 18:00:00,2013-08-18 19:00:00,2015-07-07 11:00:00,7102,7100
1,2,2012-10-02 09:00:00,2015-07-07 11:00:00,2015-07-07 12:00:00,2016-07-15 04:00:00,14202,7100
2,3,2012-10-02 09:00:00,2016-07-15 04:00:00,2016-07-15 05:00:00,2017-05-10 04:00:00,21302,7100


## 2. Baseline và quy tắc so sánh

Baseline là trung bình theo thứ–giờ, chỉ học từ train. Hai cấu hình dùng đúng cùng train, validation và fold.
Tuning chọn MAE trung bình CV thấp nhất, sau đó fit lại train và chấm validation. Giữ cùng miền tham số/ngân sách tìm kiếm giữa A và B; tham số được chọn có thể khác.
`run_config(..., raw_train=...)` học lại thống kê điền thời tiết trong mỗi fold; scaler của MLP nằm trong pipeline. Nhãn thời tiết dùng danh mục cố định trong mã dự án.
Mức thay đổi phản ánh cả việc thêm thời tiết và lựa chọn lại tham số theo cùng quy tắc. MLP có thể cảnh báo chưa hội tụ; giữ cảnh báo để xem xét kết quả.

In [3]:
baseline_prediction = weekday_hour_baseline(train, validation)
baseline_metrics = regression_metrics(validation.traffic_volume, baseline_prediction)
display(pd.Series(baseline_metrics, name='Baseline thứ–giờ'))
results = []
run_dir.mkdir(parents=True, exist_ok=False)

def save_checkpoint(item):
    """Lưu ngay từng mô hình để không mất kết quả khi cell sau gặp lỗi."""
    stem = item['config'] + '__' + item['model'].replace(' ', '_')
    pd.DataFrame([{'config': item['config'], 'model': item['model'], **item['metrics']}]).to_csv(
        run_dir / (stem + '_metrics.csv'), index=False)
    prediction = validation[['date_time', 'traffic_volume']].copy()
    prediction['prediction'] = item['prediction']
    prediction.to_csv(run_dir / (stem + '_predictions.csv'), index=False)
    item['search'].scores.to_csv(run_dir / (stem + '_cv.csv'), index=False)
    (run_dir / (stem + '_params.json')).write_text(
        json.dumps(item['search'].best_params_, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

MAE     285.027201
RMSE    501.407163
R2        0.934937
Name: Baseline thứ–giờ, dtype: float64

## 3. Huấn luyện và tuning — từng mô hình với A rồi B
Các cell dưới thực sự huấn luyện. Chạy tuần tự từ đầu; mỗi lần chạy tạo thư mục mới, không ghi đè bản cũ.

### Linear Regression

In [4]:
model_name = 'Linear Regression'
for config, features in FEATURE_SETS.items():
    print('Đang chạy', config, model_name, flush=True)
    item, = run_config(config, features, train, validation, [model_name], raw_train=splits['train'])
    results = [r for r in results if (r['config'], r['model']) != (config, model_name)]
    results.append(item)
    save_checkpoint(item)
    display(pd.DataFrame([{'config': config, 'model': model_name, **item['metrics']}]))

Đang chạy A_time Linear Regression


,config,model,MAE,RMSE,R2
0,A_time,Linear Regression,1602.862316,1811.508351,0.150756


Đang chạy B_time_weather Linear Regression


,config,model,MAE,RMSE,R2
0,B_time_weather,Linear Regression,1557.047833,1776.471458,0.183289


### Random Forest

In [5]:
model_name = 'Random Forest'
for config, features in FEATURE_SETS.items():
    print('Đang chạy', config, model_name, flush=True)
    item, = run_config(config, features, train, validation, [model_name], raw_train=splits['train'])
    results = [r for r in results if (r['config'], r['model']) != (config, model_name)]
    results.append(item)
    save_checkpoint(item)
    display(pd.DataFrame([{'config': config, 'model': model_name, **item['metrics']}]))

Đang chạy A_time Random Forest


,config,model,MAE,RMSE,R2
0,A_time,Random Forest,285.112954,501.423348,0.934933


Đang chạy B_time_weather Random Forest


,config,model,MAE,RMSE,R2
0,B_time_weather,Random Forest,274.173957,477.407717,0.941016


### Gradient Boosting

In [6]:
model_name = 'Gradient Boosting'
for config, features in FEATURE_SETS.items():
    print('Đang chạy', config, model_name, flush=True)
    item, = run_config(config, features, train, validation, [model_name], raw_train=splits['train'])
    results = [r for r in results if (r['config'], r['model']) != (config, model_name)]
    results.append(item)
    save_checkpoint(item)
    display(pd.DataFrame([{'config': config, 'model': model_name, **item['metrics']}]))

Đang chạy A_time Gradient Boosting


,config,model,MAE,RMSE,R2
0,A_time,Gradient Boosting,299.90858,513.989966,0.931631


Đang chạy B_time_weather Gradient Boosting


,config,model,MAE,RMSE,R2
0,B_time_weather,Gradient Boosting,288.845351,492.965275,0.93711


### MLP

In [ ]:
model_name = 'MLP'
for config, features in FEATURE_SETS.items():
    print('Đang chạy', config, model_name, flush=True)
    item, = run_config(config, features, train, validation, [model_name], raw_train=splits['train'])
    results = [r for r in results if (r['config'], r['model']) != (config, model_name)]
    results.append(item)
    save_checkpoint(item)
    display(pd.DataFrame([{'config': config, 'model': model_name, **item['metrics']}]))

Đang chạy A_time MLP


e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,config,model,MAE,RMSE,R2
0,A_time,MLP,417.475287,617.723406,0.901249


Đang chạy B_time_weather MLP


e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMa

### XGBoost

In [ ]:
model_name = 'XGBoost'
for config, features in FEATURE_SETS.items():
    print('Đang chạy', config, model_name, flush=True)
    item, = run_config(config, features, train, validation, [model_name], raw_train=splits['train'])
    results = [r for r in results if (r['config'], r['model']) != (config, model_name)]
    results.append(item)
    save_checkpoint(item)
    display(pd.DataFrame([{'config': config, 'model': model_name, **item['metrics']}]))

## 4. So sánh B với A trên validation
Giảm MAE dương nghĩa là B tốt hơn A. Không lấy kết quả validation này làm kết luận test cuối.

In [ ]:
assert {(r['config'], r['model']) for r in results} == {(c,m) for c in FEATURE_SETS for m in MODEL_NAMES}
metrics = results_table(results)
a = metrics.loc[metrics.config.eq('A_time')].set_index('model')
b = metrics.loc[metrics.config.eq('B_time_weather')].set_index('model')
comparison = pd.DataFrame(index=pd.Index(MODEL_NAMES, name='model'))
for metric in ('MAE', 'RMSE', 'R2'):
    comparison[metric + '_A'] = a[metric]
    comparison[metric + '_B'] = b[metric]
comparison['MAE_gain'] = comparison.MAE_A - comparison.MAE_B
comparison['MAE_gain_pct'] = np.where(comparison.MAE_A.ne(0), 100 * comparison.MAE_gain / comparison.MAE_A, np.nan)
display(comparison.round(3))
print('Baseline thứ–giờ:', baseline_metrics)
print('Các kết quả chỉ dùng', len(validation), 'giờ validation; test chưa đánh giá.')

In [ ]:
import matplotlib.pyplot as plt
ax = comparison[['MAE_A', 'MAE_B']].plot.bar(figsize=(10, 4), rot=15, color=['#627d98', '#168a78'])
ax.axhline(baseline_metrics['MAE'], color='#a85428', linestyle='--', label='Baseline thứ–giờ')
ax.set_ylabel('MAE (xe/giờ)')
ax.set_title('A: giờ + thứ | B: A + thời tiết — cùng validation')
ax.legend()
plt.tight_layout()
plt.show()
ax.figure.savefig(run_dir / 'comparison_mae.png', dpi=150, bbox_inches='tight')